## Model selection

### Three state-of-the-art algorithms
* Gradient Boosting Clasifier
* Tensorflow Neural Network
* Pythorch Neural Network

In [1]:
import numpy as np
import pandas as pd
import random


from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from keras_tuner import RandomSearch

import torch
import torch.nn as nn
import torch.optim as optim


#set random seed
my_seed=42
np.random.seed(my_seed)

In [2]:
# Data set load
data=pd.read_csv('./data/C201905_counterfactuals/simulation_C201905_counterfactuals__ev0.1812.csv',index_col=0)


# Create a target variable, 'delay' and remove unnecessary columns
data['delay']=(data['actual_duration']>data['baseline_duration'])*1
data.drop(columns=['actual_duration','baseline_duration','critical_path'],inplace=True)
# Sort the columns
columns_sorted = sorted(data.columns, key=lambda x: int(x.replace('duration', '')) if 'duration' in x else float('inf'))
data = data[columns_sorted]
data.head(2)

,duration1,duration2,duration3,duration4,duration5,duration6,duration7,duration8,duration9,duration10,duration11,duration12,duration13,duration14,duration15,delay
0,1738.595774,1967.122966,256.504345,2186.408241,1723.181771,1898.648645,1795.289433,1599.760892,2224.382275,2489.048379,2124.279759,139.271436,8.498099,234.542254,7.655393,1
1,1717.864331,1881.352898,266.048082,2324.022291,1849.728508,2083.309016,1839.782891,1593.929687,2323.598219,2279.745524,2334.495021,145.448900,8.235859,238.908956,8.357302,1


In [3]:
# Normalize the dataset to improve the performance of the classifiers

def normalize_dataframe(df, feature_range=(0, 1)):
    """
    Normaliza un DataFrame de pandas utilizando MinMaxScaler.

    Args:
        df (pd.DataFrame): DataFrame a normalizar.
        feature_range (tuple): Rango de valores para la normalización (por defecto (0, 1)).

    Returns:
        pd.DataFrame: DataFrame normalizado.
        MinMaxScaler: Escalador utilizado para la normalización.
    """
    scaler = MinMaxScaler(feature_range=feature_range)
    normalized_data = scaler.fit_transform(df)
    normalized_df = pd.DataFrame(normalized_data, columns=df.columns, index=df.index)
    return normalized_df, scaler

def denormalize_dataframe(normalized_df, scaler):
    """
    Desnormaliza un DataFrame de pandas utilizando un MinMaxScaler previamente ajustado.

    Args:
        normalized_df (pd.DataFrame): DataFrame normalizado.
        scaler (MinMaxScaler): Escalador utilizado para la normalización.

    Returns:
        pd.DataFrame: DataFrame desnormalizado.
    """
    denormalized_data = scaler.inverse_transform(normalized_df)
    denormalized_df = pd.DataFrame(denormalized_data, columns=normalized_df.columns, index=normalized_df.index)
    return denormalized_df


data_to_normalize = data.iloc[:, :-1]  # Select all columns except the last one
normalized_data, my_scaler = normalize_dataframe(data_to_normalize)
data = pd.concat([normalized_data, data.iloc[:, -1]], axis=1)  # Combine normalized data with the last column

data.head(2)


,duration1,duration2,duration3,duration4,duration5,duration6,duration7,duration8,duration9,duration10,duration11,duration12,duration13,duration14,duration15,delay
0,0.751643,0.821171,0.653332,0.547362,0.393468,0.432562,0.715566,0.442461,0.804429,0.891496,0.551165,0.700622,0.641918,0.698465,0.380315,1
1,0.675055,0.594889,0.759338,0.713173,0.582989,0.679778,0.789338,0.432862,0.938094,0.637179,0.819892,0.830761,0.556337,0.752252,0.604439,1


In [4]:
# Split the dataset into features and class
X = data.drop(columns=['delay'])
y = data['delay']

# "stratify=y is used in train_test_split to ensure that the class distribution remains the same in both the training and test sets."
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=my_seed, stratify=y)

# Verify the split
print("Unique labels in training:", np.unique(y_train, return_counts=True))
print("Unique labels in test:", np.unique(y_test, return_counts=True))

Unique labels in training: (array([0, 1]), array([ 5564, 14436]))
Unique labels in test: (array([0, 1]), array([1391, 3609]))


### Gradient Boosting Clasifier

In [5]:
# Results dataframe
results = pd.DataFrame([],columns=['model','kf','Accuracy'])

# Stratified K-Fold cross-validation
kfold =StratifiedKFold(n_splits=5, random_state=my_seed,shuffle=True)

In [6]:
model='GradientBoostingC'
k=0
for train_index, test_index in kfold.split(X,y):

  # parameter optimization in the inner folds
    # n_estimators: number of  boosting stages
    # max_depth of the regression estimators
  mdc = GridSearchCV(GradientBoostingClassifier(),
                    param_grid={"n_estimators": np.linspace(50,200,10).astype('int'),  
                                "max_depth": np.linspace(1,10,5).astype('int')},
                    n_jobs=6)  
  mdc.fit(X.iloc[train_index,:], y.iloc[train_index])
  
  # score of the best model in the outer fold
  results = results.append( pd.DataFrame([[model,k, accuracy_score(y.iloc[test_index],mdc.best_estimator_.predict(X.iloc[test_index,:]))]],columns=['model','kf','Accuracy']))
  k+=1

/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_6716/1260766298.py:15: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append( pd.DataFrame([[model,k, accuracy_score(y.iloc[test_index],mdc.best_estimator_.predict(X.iloc[test_index,:]))]],columns=['model','kf','Accuracy']))
/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_6716/1260766298.py:15: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append( pd.DataFrame([[model,k, accuracy_score(y.iloc[test_index],mdc.best_estimator_.predict(X.iloc[test_index,:]))]],columns=['model','kf','Accuracy']))
/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_6716/1260766298.py:15: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  res

### Tensorflow neural network

In [7]:
def build_model(hp):
    model = keras.Sequential([
        layers.Dense(hp.Int('units', min_value=32, max_value=128, step=32), activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        layers.Dense(2, activation='softmax')
    ])
    model.compile(optimizer=keras.optimizers.Adam(hp.Choice('learning_rate', [0.01, 0.001, 0.0001])),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model='Tensorflow'
k=0
for train_index, test_index in kfold.split(X,y):

  # parameter optimization in the inner folds
    # units: number of  neurons in the hidden layer
    # learning_rate
  tuner = RandomSearch(build_model, objective='val_accuracy', max_trials=10, executions_per_trial=1, directory='tuner_dir')

  tuner.search(X.iloc[train_index,:], y.iloc[train_index], epochs=30, validation_split=0.2, batch_size=32) 
  best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
  best_model = tuner.hypermodel.build(best_hps)
  best_model.fit(X.iloc[train_index,:], y.iloc[train_index], epochs=30, batch_size=32, validation_split=0.2, callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
  test_loss, test_acc = best_model.evaluate(X.iloc[test_index,:], y.iloc[test_index])
  
  # score of the best model in the outer fold
  results = results.append( pd.DataFrame([[model,k, test_acc]],columns=['model','kf','Accuracy']))
  k+=1

Reloading Tuner from tuner_dir/untitled_project/tuner0.json
Epoch 1/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 560us/step - accuracy: 0.7084 - loss: 0.6897 - val_accuracy: 0.8455 - val_loss: 0.4226
Epoch 2/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 432us/step - accuracy: 0.8411 - loss: 0.4046 - val_accuracy: 0.8393 - val_loss: 0.3644
Epoch 3/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 440us/step - accuracy: 0.8521 - loss: 0.3579 - val_accuracy: 0.8490 - val_loss: 0.3393
Epoch 4/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 436us/step - accuracy: 0.8621 - loss: 0.3292 - val_accuracy: 0.8640 - val_loss: 0.3299
Epoch 5/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 428us/step - accuracy: 0.8638 - loss: 0.3158 - val_accuracy: 0.8620 - val_loss: 0.3124
Epoch 6/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 432us/step - accuracy: 0.8659 - loss: 0.3118 - val_accuracy: 0.8620 - val_loss: 0.3061
Epoch 7/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 428us/step - accuracy: 0.8679 - loss: 0.3035 - val_accuracy: 0.8668 - val_loss: 0.2992
Epoch 8/30
500/500 ━━━━━━━━━━━

/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_6716/2173360707.py:27: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append( pd.DataFrame([[model,k, test_acc]],columns=['model','kf','Accuracy']))


500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 619us/step - accuracy: 0.7325 - loss: 0.6633 - val_accuracy: 0.8207 - val_loss: 0.4211
Epoch 2/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 495us/step - accuracy: 0.8421 - loss: 0.4029 - val_accuracy: 0.8600 - val_loss: 0.3704
Epoch 3/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 498us/step - accuracy: 0.8571 - loss: 0.3468 - val_accuracy: 0.8553 - val_loss: 0.3324
Epoch 4/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 432us/step - accuracy: 0.8672 - loss: 0.3244 - val_accuracy: 0.8577 - val_loss: 0.3590
Epoch 5/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 432us/step - accuracy: 0.8643 - loss: 0.3183 - val_accuracy: 0.8650 - val_loss: 0.3077
Epoch 6/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 431us/step - accuracy: 0.8738 - loss: 0.3021 - val_accuracy: 0.8677 - val_loss: 0.3035
Epoch 7/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 433us/step - accuracy: 0.8735 - loss: 0.2983 - val_accuracy: 0.8705 - val_loss: 0.2963
Epoch 8/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 452us/step - accuracy: 0.8757 - loss: 0.2899 - val_accurac

/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_6716/2173360707.py:27: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append( pd.DataFrame([[model,k, test_acc]],columns=['model','kf','Accuracy']))


500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 552us/step - accuracy: 0.7401 - loss: 0.6466 - val_accuracy: 0.8503 - val_loss: 0.4447
Epoch 2/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 434us/step - accuracy: 0.8357 - loss: 0.4076 - val_accuracy: 0.8687 - val_loss: 0.3613
Epoch 3/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 427us/step - accuracy: 0.8577 - loss: 0.3539 - val_accuracy: 0.8727 - val_loss: 0.3268
Epoch 4/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 436us/step - accuracy: 0.8606 - loss: 0.3339 - val_accuracy: 0.8668 - val_loss: 0.3118
Epoch 5/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 434us/step - accuracy: 0.8608 - loss: 0.3227 - val_accuracy: 0.8770 - val_loss: 0.2977
Epoch 6/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 429us/step - accuracy: 0.8716 - loss: 0.3059 - val_accuracy: 0.8765 - val_loss: 0.3183
Epoch 7/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 429us/step - accuracy: 0.8758 - loss: 0.2959 - val_accuracy: 0.8848 - val_loss: 0.2926
Epoch 8/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 430us/step - accuracy: 0.8757 - loss: 0.2954 - val_accurac

/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_6716/2173360707.py:27: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append( pd.DataFrame([[model,k, test_acc]],columns=['model','kf','Accuracy']))


500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 671us/step - accuracy: 0.7339 - loss: 0.6723 - val_accuracy: 0.8460 - val_loss: 0.4213
Epoch 2/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 436us/step - accuracy: 0.8373 - loss: 0.4115 - val_accuracy: 0.8615 - val_loss: 0.3642
Epoch 3/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 433us/step - accuracy: 0.8591 - loss: 0.3595 - val_accuracy: 0.8695 - val_loss: 0.3386
Epoch 4/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 441us/step - accuracy: 0.8600 - loss: 0.3402 - val_accuracy: 0.8665 - val_loss: 0.3191
Epoch 5/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 454us/step - accuracy: 0.8649 - loss: 0.3192 - val_accuracy: 0.8633 - val_loss: 0.3469
Epoch 6/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 446us/step - accuracy: 0.8630 - loss: 0.3201 - val_accuracy: 0.8615 - val_loss: 0.3136
Epoch 7/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 436us/step - accuracy: 0.8714 - loss: 0.3051 - val_accuracy: 0.8662 - val_loss: 0.3013
Epoch 8/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 436us/step - accuracy: 0.8686 - loss: 0.3032 - val_accurac

/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_6716/2173360707.py:27: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append( pd.DataFrame([[model,k, test_acc]],columns=['model','kf','Accuracy']))


500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 550us/step - accuracy: 0.7456 - loss: 0.6526 - val_accuracy: 0.8450 - val_loss: 0.4094
Epoch 2/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 437us/step - accuracy: 0.8456 - loss: 0.3934 - val_accuracy: 0.8660 - val_loss: 0.3569
Epoch 3/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 434us/step - accuracy: 0.8617 - loss: 0.3451 - val_accuracy: 0.8658 - val_loss: 0.3289
Epoch 4/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 432us/step - accuracy: 0.8679 - loss: 0.3194 - val_accuracy: 0.8317 - val_loss: 0.3687
Epoch 5/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 435us/step - accuracy: 0.8664 - loss: 0.3143 - val_accuracy: 0.8568 - val_loss: 0.3154
Epoch 6/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 440us/step - accuracy: 0.8666 - loss: 0.3120 - val_accuracy: 0.8643 - val_loss: 0.3117
Epoch 7/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 433us/step - accuracy: 0.8768 - loss: 0.2952 - val_accuracy: 0.8687 - val_loss: 0.3000
Epoch 8/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 432us/step - accuracy: 0.8675 - loss: 0.3036 - val_accurac

/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_6716/2173360707.py:27: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append( pd.DataFrame([[model,k, test_acc]],columns=['model','kf','Accuracy']))


### Pytorch

In [8]:
# Data preparation

def safe_to_tensor(data_numpy, batch_size=100): # Adjust batch size as needed
    """Convierte arrays NumPy a tensores PyTorch por lotes"""
    if not isinstance(data_numpy, np.ndarray):
        raise ValueError("Input must be a NumPy array")
    
    tensor_parts = []
    for i in range(0, len(data_numpy), batch_size):
        batch = data_numpy[i:i+batch_size]
        tensor_parts.append(torch.from_numpy(batch).float())
    return torch.cat(tensor_parts)


# Convert to tensors
X_train_tensor = safe_to_tensor(X_train.values.astype(np.float32))
X_test_tensor = safe_to_tensor(X_test.values.astype(np.float32))

# Label tensors
y_train_tensor = torch.tensor(y_train.values.astype(np.float32)).unsqueeze(1)
y_test_tensor = torch.tensor(y_test.values.astype(np.float32)).unsqueeze(1)

# Verify tensor shapes
print("Tensors successfully created:")
print(f"X_train_tensor shape: {X_train_tensor.shape}")
print(f"y_train_tensor shape: {y_train_tensor.shape}")

Tensors successfully created:
X_train_tensor shape: torch.Size([20000, 15])
y_train_tensor shape: torch.Size([20000, 1])


In [9]:
# Model definition
class BinaryClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=32):
        super(BinaryClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()  # Capa sigmoide explícita
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return self.sigmoid(x)  # Devuelve tensor con sigmoide aplicada
    

# Model instantiation
model = BinaryClassifier(input_dim=X_train.shape[1])
criterion = nn.BCELoss()  # Ahora usamos BCELoss porque aplicamos sigmoide manualmente
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [10]:
# Training loop with memory management
from tqdm import tqdm  # Progress bar

# Definir el tamaño del batch reducido
batch_size = 100  

# 3. Entrenamiento con manejo de memoria y lotes
try:
    model.train()
    for epoch in tqdm(range(30), desc="Entrenando"):
        optimizer.zero_grad()
        
        # Entrenamiento por lotes (batches)
        for i in range(0, len(X_train_tensor), batch_size):
            batch_X = X_train_tensor[i:i + batch_size]
            batch_y = y_train_tensor[i:i + batch_size]

            # Forward pass por lote
            outputs = model(batch_X)
            
            # Verificación de formas
            if outputs.shape != batch_y.shape:
                print(f"Error: outputs shape {outputs.shape} != y shape {batch_y.shape}")
                break

            loss = criterion(outputs, batch_y)
            
            # Backpropagation con manejo de gradientes
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Evita gradientes explosivos
            optimizer.step()
        
        if (epoch + 1) % 5 == 0:
            print(f'Epoch {epoch + 1}, Loss: {loss.item():.4f}')
            
except RuntimeError as e:
    print(f"Error durante el entrenamiento: {str(e)}")
    print("Posibles soluciones:")
    print("1. Reducir el tamaño del batch")
    print("2. Usar float32 en lugar de float64")
    print("3. Verificar dimensiones de entrada")


Entrenando:  33%|███▎      | 10/30 [00:00<00:00, 21.04it/s]

Epoch 5, Loss: 0.2974
Epoch 10, Loss: 0.2470


Entrenando:  63%|██████▎   | 19/30 [00:00<00:00, 24.28it/s]

Epoch 15, Loss: 0.1963
Epoch 20, Loss: 0.1481


Entrenando: 100%|██████████| 30/30 [00:01<00:00, 22.40it/s]

Epoch 25, Loss: 0.1392
Epoch 30, Loss: 0.1439


In [11]:
# Change to eval mode
model.eval()
batch_size = 100  

# Use torch.no_grad() during inference (evaluation) to reduce memory usage and speed up computations by skipping gradient tracking
with torch.no_grad():
    correct_preds = 0
    total_preds = 0
    
    # Batch processing for evaluation
    for i in tqdm(range(0, len(X_test_tensor), batch_size), desc="Evaluando"):
        batch_X = X_test_tensor[i:i + batch_size]
        batch_y = y_test_tensor[i:i + batch_size]
        
        # Predictions as probabilities
        batch_output = model(batch_X)
        
        # Predicciones as classes
        batch_preds = (batch_output > 0.5).float()
        
        # Count to compute accuracy
        correct_preds += (batch_preds == batch_y).sum().item()
        total_preds += batch_y.size(0)

    # Accuracy calculation
    pt_accuracy = correct_preds / total_preds
    print(f'Accuracy: {pt_accuracy:.4f}')

Evaluando: 100%|██████████| 50/50 [00:00<00:00, 10841.36it/s]

Accuracy: 0.9440


In [12]:
# I do not run a cross-validation for the PyTorch model because it is very slow
# Save the results

model_name = "Pytorch"
n_repeats = 5

# Create DataFrame in one step
results_pt = pd.DataFrame({
    'model': [model_name] * n_repeats,
    'kf': [0] * n_repeats, 
    'Accuracy': [pt_accuracy] * n_repeats
})

print(results_pt)

results=results.append(results_pt, ignore_index=True)

     model  kf  Accuracy
0  Pytorch   0     0.944
1  Pytorch   0     0.944
2  Pytorch   0     0.944
3  Pytorch   0     0.944
4  Pytorch   0     0.944


/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_6716/3957595904.py:16: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results=results.append(results_pt, ignore_index=True)


### Comparison

In [13]:
results.groupby('model').agg({'Accuracy':['mean','std']}).sort_values(by=('Accuracy', 'mean'),ascending=False)


Accuracy          
                      mean       std
model                               
GradientBoostingC  0.98784  0.001307
Tensorflow         0.97688  0.008339
Pytorch            0.94400  0.000000

### Model hiperparameter tunning and model training

In [14]:
mdc = GridSearchCV(GradientBoostingClassifier(),
    param_grid={"n_estimators": np.linspace(50,200,10).astype('int'),  
                "max_depth": np.linspace(1,10,5).astype('int')},
    n_jobs=8) 
mdc.fit(X, y)
mdc.best_params_ # {'max_depth': 5, 'n_estimators': 100}

# Save the model
import joblib
joblib.dump(mdc.best_estimator_, 'gbc_best_model.pkl')
# Load the model
loaded_model = joblib.load('gbc_best_model.pkl')

In [15]:
tuner = RandomSearch(build_model, objective='val_accuracy', max_trials=10, executions_per_trial=1, directory='tuner_dir')
tuner.search(X, y, epochs=30, validation_split=0.2, batch_size=32)
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"Best TensorFlow Model: units={best_hps.get('units')}, learning_rate={best_hps.get('learning_rate')}")

# Save the model
best_model.save('tf_best_model.keras')
# Load the model
loaded_model = keras.models.load_model('tf_best_model.keras')

Reloading Tuner from tuner_dir/untitled_project/tuner0.json
Best TensorFlow Model: units=128, learning_rate=0.001


In [16]:
# Tensors
X_tensor = safe_to_tensor(X.values.astype(np.float32))
y_tensor = torch.tensor(y.values.astype(np.float32)).unsqueeze(1)


try:
    model.train()
    for epoch in tqdm(range(30), desc="Entrenando"):
        optimizer.zero_grad()
        
        # Entrenamiento por lotes (batches)
        for i in range(0, len(X_train_tensor), batch_size):
            batch_X = X_tensor[i:i + batch_size]
            batch_y = y_tensor[i:i + batch_size]

            # Forward pass por lote
            outputs = model(batch_X)
            
            # Verificación de formas
            if outputs.shape != batch_y.shape:
                print(f"Error: outputs shape {outputs.shape} != y shape {batch_y.shape}")
                break

            loss = criterion(outputs, batch_y)
            
            # Backpropagation con manejo de gradientes
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Evita gradientes explosivos
            optimizer.step()
        
        if (epoch + 1) % 5 == 0:
            print(f'Epoch {epoch + 1}, Loss: {loss.item():.4f}')
except RuntimeError as e:
    print(f"Error durante el entrenamiento: {str(e)}")
    print("Posibles soluciones:")
    print("1. Reducir el tamaño del batch")
    print("2. Usar float32 en lugar de float64")
    print("3. Verificar dimensiones de entrada")

#Save model
torch.save(model.state_dict(), 'pytorch_best_model.pth')
#Load model
model = BinaryClassifier(input_dim=X_train.shape[1])
model.load_state_dict(torch.load('pytorch_best_model.pth'))

Entrenando:  30%|███       | 9/30 [00:00<00:00, 21.66it/s]

Epoch 5, Loss: 0.1535
Epoch 10, Loss: 0.1544


Entrenando:  60%|██████    | 18/30 [00:00<00:00, 24.27it/s]

Epoch 15, Loss: 0.1569
Epoch 20, Loss: 0.0854


Entrenando:  90%|█████████ | 27/30 [00:01<00:00, 24.27it/s]

Epoch 25, Loss: 0.0926


Entrenando: 100%|██████████| 30/30 [00:01<00:00, 22.94it/s]

Epoch 30, Loss: 0.0702


<All keys matched successfully>